# Boosted Test Trace and Answer Analysis

This notebook analyzes boosted **inference runs only**. It never loads or joins mined trace datasets.

It discovers results under:

- `outputs/inference/test_oracle_boosted`
- `outputs/inference/current_lvar_model_boosted`

The analysis preserves the inference-focused functionality of `test_trace_answer_analysis.ipynb`: accuracy, correct-vs-incorrect behavior, decoding entropy, qualitative errors, domain/topic breakdowns, and table export. It additionally analyzes the boost target, layer mode, alpha sweep, trace attention mass, visual trace attention mass, and THINK attention mass.


In [ ]:
%matplotlib inline

import json
import math
from pathlib import Path
import re

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/csalt/Haider/DVLM/lvar"),
]
ROOT = next(
    (
        candidate.resolve()
        for candidate in ROOT_CANDIDATES
        if (
            (candidate / "outputs/inference/test_oracle_boosted").exists()
            or (candidate / "outputs/inference/current_lvar_model_boosted").exists()
        )
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate a repository containing boosted inference outputs. "
        "Run this notebook from the repository or configure ROOT_CANDIDATES."
    )

ORACLE_ROOT = ROOT / "outputs/inference/test_oracle_boosted"
FULL_LVAR_ROOT = ROOT / "outputs/inference/current_lvar_model_boosted"

print(f"ROOT: {ROOT}")
print(f"ORACLE_ROOT: {ORACLE_ROOT} (exists={ORACLE_ROOT.exists()})")
print(f"FULL_LVAR_ROOT: {FULL_LVAR_ROOT} (exists={FULL_LVAR_ROOT.exists()})")


## Helpers

These helpers load JSON/JSONL safely, recover boosted-run metadata from both summaries and directory names, and normalize full-pipeline and oracle runs into one schema.


In [ ]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path):
    rows = []
    bad_rows = []
    path = Path(path)
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                bad_rows.append({
                    "path": str(path),
                    "line_number": line_number,
                    "error": str(exc),
                    "context": stripped[max(0, exc.pos - 100): exc.pos + 100],
                })
    return rows, bad_rows


def clean_ckpt_name(value):
    if value is None:
        return None
    value = str(value)
    return value.replace("mined_by_", "").replace("evaluated_by_", "").replace("_ckpt", "")


def path_value(parts, prefix):
    value = next((part[len(prefix):] for part in parts if part.startswith(prefix)), None)
    return value


def parse_boosted_path(path):
    parts = Path(path).parts
    alpha_value = path_value(parts, "alpha_")
    return {
        "mined_by": next((clean_ckpt_name(p) for p in parts if p.startswith("mined_by_")), None),
        "evaluated_by": next((clean_ckpt_name(p) for p in parts if p.startswith("evaluated_by_")), None),
        "trace_variant": path_value(parts, "trace_variant_"),
        "boost_target": path_value(parts, "target_"),
        "layer_mode": path_value(parts, "layers_"),
        "alpha": float(alpha_value) if alpha_value is not None else np.nan,
    }


def parse_mining_and_replay_context(summary, artifact_path):
    trace_name = Path(summary.get("trace_path", "")).name
    trace_match = re.search(r"_(global|coarse)\.jsonl$", trace_name)
    mining_context = trace_match.group(1) if trace_match else None
    suffix = re.search(r"_replayed-under_(global|coarse|full_context|global_mean)(?:_|\.)", Path(artifact_path).name)
    replay_context = suffix.group(1) if suffix else mining_context
    return mining_context, replay_context


def resolve_artifact(path_value):
    if not path_value:
        return None
    path = Path(path_value)
    if path.is_absolute():
        return path
    return ROOT / path


def pretty_variant(value):
    mapping = {
        "raw": "raw",
        "filtered_cap": "filtered (cap)",
        "filtered_no_cap": "filtered (no cap)",
        "controller_generated": "controller-generated",
    }
    return mapping.get(str(value), str(value))


RUN_KEYS = [
    "run_type", "mined_by", "context", "replay_context", "evaluated_by", "trace_variant",
    "boost_target", "layer_mode", "alpha",
]
MASS_METRICS = [
    "trace_attention_mass",
    "visual_trace_attention_mass",
    "think_attention_mass",
]
ENTROPY_METRICS = [
    "answer_option_entropy",
    "decoded_token_entropy_mean",
    "decoded_token_entropy_median",
    "decoded_token_entropy_max",
]


def make_run_id(record):
    return " | ".join(str(record.get(key)) for key in RUN_KEYS)


def format_alpha(value):
    return "NA" if pd.isna(value) else f"{float(value):.1f}"


## 1. Discover Boosted Inference Runs

Each summary contributes accuracy, output-length statistics, boost configuration, and run-level attention masses. Metadata is read from the summary first and falls back to the directory hierarchy.


In [ ]:
summary_rows = []
bad_summary_rows = []

for run_type, inference_root in [
    ("oracle", ORACLE_ROOT),
    ("full_lvar", FULL_LVAR_ROOT),
]:
    if not inference_root.exists():
        continue
    for summary_path in sorted(inference_root.rglob("*_summary.json")):
        try:
            summary = load_json(summary_path)
        except Exception as exc:
            bad_summary_rows.append({"path": str(summary_path), "error": repr(exc)})
            continue
        metrics = summary.get("metrics", {}) or {}
        if not metrics:
            continue
        parsed = parse_boosted_path(summary_path)
        boost = summary.get("trace_boost", {}) or {}
        if not bool(boost.get("enabled", True)):
            continue

        if run_type == "full_lvar":
            mined_by = "full_lvar"
            evaluated_by = "lvar+controller"
            context = "coarse" if summary.get("coarse_context") else "global"
            replay_context = context
            trace_variant = "controller_generated"
        else:
            mined_by = parsed["mined_by"]
            evaluated_by = parsed["evaluated_by"]
            context, replay_context = parse_mining_and_replay_context(summary, summary_path)
            trace_variant = summary.get("trace_variant") or parsed["trace_variant"]

        record = {
            "run_type": run_type,
            "mined_by": mined_by,
            "context": context,
            "replay_context": replay_context,
            "evaluated_by": evaluated_by,
            "trace_variant": trace_variant,
            "trace_variant_label": pretty_variant(trace_variant),
            "boost_target": boost.get("target") or parsed["boost_target"],
            "layer_mode": boost.get("layer_mode") or parsed["layer_mode"],
            "alpha": boost.get("alpha", parsed["alpha"]),
            "apply_stage": boost.get("apply_stage"),
            "accuracy": metrics.get("accuracy"),
            "correct": metrics.get("correct"),
            "total": metrics.get("total") or summary.get("num_examples"),
            "avg_trace_actions": metrics.get("avg_trace_actions", metrics.get("avg_controller_actions")),
            "avg_output_tokens": metrics.get("avg_output_tokens"),
            "trace_attention_mass": metrics.get("trace_attention_mass"),
            "visual_trace_attention_mass": metrics.get("visual_trace_attention_mass"),
            "think_attention_mass": metrics.get("think_attention_mass"),
            "output_path": summary.get("output_path"),
            "entropy_tracking_path": summary.get("entropy_tracking_path"),
            "summary_path": str(summary_path.relative_to(ROOT)),
        }
        record["run_id"] = make_run_id(record)
        summary_rows.append(record)

run_summary = pd.DataFrame(summary_rows)
if run_summary.empty:
    raise ValueError(f"No boosted inference summaries found under {ORACLE_ROOT} or {FULL_LVAR_ROOT}")

for column in ["accuracy", "alpha", "avg_trace_actions", "avg_output_tokens", *MASS_METRICS]:
    run_summary[column] = pd.to_numeric(run_summary[column], errors="coerce")
run_summary["accuracy_pct"] = 100 * run_summary["accuracy"]
run_summary = run_summary.sort_values(RUN_KEYS).reset_index(drop=True)

print(f"Runs loaded: {len(run_summary):,}")
print(f"Bad summaries: {len(bad_summary_rows):,}")
display(run_summary.head())
display(run_summary.groupby(["run_type", "boost_target", "layer_mode"]).size().rename("runs").to_frame())


### Sweep Coverage

The intended grid contains both targets, both layer modes, and alpha values 0.1–0.5 for every base inference setting. This table makes missing or duplicate runs visible before comparing results.


In [ ]:
EXPECTED_TARGETS = ["trace_all", "trace_visual"]
EXPECTED_LAYER_MODES = ["all", "latter_half"]
EXPECTED_ALPHAS = [0.01,0.1, 0.2, 0.3, 0.4, 0.5]
EXPECTED_COMBINATIONS = {
    (target, layer_mode, alpha)
    for target in EXPECTED_TARGETS
    for layer_mode in EXPECTED_LAYER_MODES
    for alpha in EXPECTED_ALPHAS
}
BASE_SETTING_KEYS = ["run_type", "mined_by", "context", "replay_context", "evaluated_by", "trace_variant"]

coverage_rows = []
for base_key, group in run_summary.groupby(BASE_SETTING_KEYS, dropna=False):
    observed = {
        (row.boost_target, row.layer_mode, round(float(row.alpha), 1))
        for row in group.itertuples()
        if pd.notna(row.alpha)
    }
    missing = sorted(EXPECTED_COMBINATIONS - observed)
    duplicates = int(group.duplicated(["boost_target", "layer_mode", "alpha"]).sum())
    record = dict(zip(BASE_SETTING_KEYS, base_key))
    record.update({
        "observed_settings": len(observed),
        "expected_settings": len(EXPECTED_COMBINATIONS),
        "missing_settings": missing,
        "duplicate_rows": duplicates,
        "complete": not missing and duplicates == 0,
    })
    coverage_rows.append(record)

sweep_coverage = pd.DataFrame(coverage_rows)
display(sweep_coverage)
if not sweep_coverage["complete"].all():
    print("Warning: at least one base inference setting has an incomplete or duplicated boost sweep.")


## 2. Accuracy Across Boost Settings

The first table ranks every run. The selector then plots alpha-response curves for one full-pipeline or oracle base setting.


In [ ]:
ranked_runs = run_summary[
    [
        *RUN_KEYS, "accuracy_pct", "correct", "total", "avg_trace_actions",
        "avg_output_tokens", *MASS_METRICS, "summary_path",
    ]
].sort_values("accuracy_pct", ascending=False).reset_index(drop=True)

display(
    ranked_runs.style.format({
        "accuracy_pct": "{:.2f}",
        "alpha": "{:.2f}",
        "avg_trace_actions": "{:.2f}",
        "avg_output_tokens": "{:.2f}",
        **{metric: "{:.5f}" for metric in MASS_METRICS},
    }).set_caption("Boosted inference settings ranked by accuracy")
)


In [ ]:
# Change these selectors to inspect a different base inference setting.
SELECT_RUN_TYPE = "full_lvar"
SELECT_MINED_BY = "full_lvar"
SELECT_CONTEXT = "global"
SELECT_REPLAY_CONTEXT = "global"
SELECT_EVALUATED_BY = "lvar+controller"
SELECT_TRACE_VARIANT = "controller_generated"

selected_summary = run_summary[
    (run_summary["run_type"] == SELECT_RUN_TYPE)
    & (run_summary["mined_by"] == SELECT_MINED_BY)
    & (run_summary["context"] == SELECT_CONTEXT)
    & (run_summary["replay_context"] == SELECT_REPLAY_CONTEXT)
    & (run_summary["evaluated_by"] == SELECT_EVALUATED_BY)
    & (run_summary["trace_variant"] == SELECT_TRACE_VARIANT)
].copy()

if selected_summary.empty:
    print("The default selection is unavailable; using the first discovered base setting.")
    first = run_summary.iloc[0]
    selected_summary = run_summary[
        (run_summary["run_type"] == first["run_type"])
        & (run_summary["mined_by"] == first["mined_by"])
        & (run_summary["context"] == first["context"])
        & (run_summary["replay_context"] == first["replay_context"])
        & (run_summary["evaluated_by"] == first["evaluated_by"])
        & (run_summary["trace_variant"] == first["trace_variant"])
    ].copy()

SELECTED_BASE = selected_summary.iloc[0][BASE_SETTING_KEYS].to_dict()
print("Selected base setting:", SELECTED_BASE)
display(selected_summary.sort_values(["boost_target", "layer_mode", "alpha"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, target in zip(axes, EXPECTED_TARGETS):
    target_frame = selected_summary[selected_summary["boost_target"] == target]
    for layer_mode, group in target_frame.groupby("layer_mode"):
        group = group.sort_values("alpha")
        ax.plot(group["alpha"], group["accuracy_pct"], marker="o", linewidth=2, label=layer_mode)
    ax.set_title(target)
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticks(EXPECTED_ALPHAS)
    ax.legend(title="Layer mode")
plt.suptitle("Accuracy response to trace-attention boosting")
plt.tight_layout()
plt.show()


In [ ]:
accuracy_ablation = (
    run_summary
    .pivot_table(
        index=BASE_SETTING_KEYS + ["alpha"],
        columns=["boost_target", "layer_mode"],
        values="accuracy_pct",
        aggfunc="first",
    )
    .sort_index()
)
display(accuracy_ablation.style.format("{:.2f}").set_caption("Accuracy (%) across target × layer-mode ablations"))

best_per_base = (
    run_summary.sort_values("accuracy_pct", ascending=False)
    .groupby(BASE_SETTING_KEYS, dropna=False)
    .head(1)
    [[*BASE_SETTING_KEYS, "boost_target", "layer_mode", "alpha", "accuracy_pct", *MASS_METRICS]]
    .reset_index(drop=True)
)
display(best_per_base.style.format({"alpha": "{:.1f}", "accuracy_pct": "{:.2f}", **{m: "{:.5f}" for m in MASS_METRICS}}))


## 3. Load Prediction Rows

Prediction JSONLs are loaded through their summaries, so only completed inference runs enter the sample-level analyses. No trace dataset is read.


In [ ]:
prediction_rows = []
bad_prediction_rows = []
missing_prediction_files = []

for summary in run_summary.itertuples(index=False):
    output_path = resolve_artifact(summary.output_path)
    if output_path is None:
        summary_path = ROOT / summary.summary_path
        candidates = list(summary_path.parent.glob("*.jsonl"))
        candidates = [path for path in candidates if "entropy_tracking" not in path.name]
        output_path = candidates[0] if len(candidates) == 1 else None
    if output_path is None or not output_path.exists():
        missing_prediction_files.append(str(output_path))
        continue

    rows, bad_rows = load_jsonl(output_path)
    bad_prediction_rows.extend(bad_rows)
    metadata = {key: getattr(summary, key) for key in RUN_KEYS}
    metadata["run_id"] = summary.run_id
    for row in rows:
        prediction_rows.append({
            **metadata,
            "example_id": row.get("example_id"),
            "question": row.get("question"),
            "correct": bool(row.get("correct")),
            "gold_answer": row.get("gold_answer"),
            "raw_answer": row.get("raw_answer"),
            "generated_text": row.get("generated_text"),
            "decoded_answer": row.get("decoded_answer"),
            "domain": row.get("domain"),
            "topic": row.get("topic"),
            "num_inference_trace_actions": (
                row.get("num_trace_actions")
                if row.get("num_trace_actions") is not None
                else row.get("num_steps")
            ),
            "num_output_tokens": row.get("num_output_tokens"),
            "trace_attention_mass": row.get("trace_attention_mass"),
            "visual_trace_attention_mass": row.get("visual_trace_attention_mass"),
            "think_attention_mass": row.get("think_attention_mass"),
            "prediction_path": str(output_path.relative_to(ROOT)) if output_path.is_relative_to(ROOT) else str(output_path),
        })

predictions = pd.DataFrame(prediction_rows)
if predictions.empty:
    raise ValueError("No boosted prediction JSONLs could be loaded.")

for column in ["alpha", "num_inference_trace_actions", "num_output_tokens", *MASS_METRICS]:
    predictions[column] = pd.to_numeric(predictions[column], errors="coerce")

print(f"Prediction rows: {len(predictions):,}")
print(f"Unique runs: {predictions['run_id'].nunique():,}")
print(f"Bad JSONL rows: {len(bad_prediction_rows):,}")
print(f"Missing prediction files: {len(missing_prediction_files):,}")
display(predictions.head())


## 4. Correct vs Incorrect Outcomes

This is the inference-only counterpart of the original correctness analysis. It compares output length, controller/replayed action count, and all three tracked attention masses.


In [ ]:
correctness_summary = (
    predictions
    .groupby(RUN_KEYS + ["correct"], dropna=False)
    .agg(
        examples=("example_id", "nunique"),
        mean_trace_actions=("num_inference_trace_actions", "mean"),
        mean_output_tokens=("num_output_tokens", "mean"),
        mean_trace_attention_mass=("trace_attention_mass", "mean"),
        median_trace_attention_mass=("trace_attention_mass", "median"),
        mean_visual_trace_attention_mass=("visual_trace_attention_mass", "mean"),
        mean_think_attention_mass=("think_attention_mass", "mean"),
    )
    .reset_index()
    .sort_values(RUN_KEYS + ["correct"])
)
display(
    correctness_summary.style.format({
        "alpha": "{:.1f}",
        "mean_trace_actions": "{:.2f}",
        "mean_output_tokens": "{:.2f}",
        "mean_trace_attention_mass": "{:.5f}",
        "median_trace_attention_mass": "{:.5f}",
        "mean_visual_trace_attention_mass": "{:.5f}",
        "mean_think_attention_mass": "{:.5f}",
    }).set_caption("Prediction behavior split by correct vs incorrect outcomes")
)


In [ ]:
SELECTED_RUN_ID = selected_summary.sort_values("accuracy_pct", ascending=False).iloc[0]["run_id"]
selected_predictions = predictions[predictions["run_id"] == SELECTED_RUN_ID].copy()

print(f"Selected run: {SELECTED_RUN_ID}")
print(f"Rows: {len(selected_predictions):,}")
display(selected_predictions["correct"].value_counts(dropna=False).rename("examples").to_frame())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric in zip(axes, MASS_METRICS):
    usable = selected_predictions.dropna(subset=[metric])
    if HAS_SEABORN:
        sns.histplot(
            data=usable, x=metric, hue="correct", bins=35, kde=True,
            stat="density", common_norm=False, element="step", ax=ax,
        )
    else:
        for correct, group in usable.groupby("correct"):
            ax.hist(group[metric], bins=35, alpha=0.45, density=True, label=f"correct={correct}")
        ax.legend()
    ax.set_title(metric.replace("_", " "))
plt.suptitle("Attention mass by correctness for the selected run")
plt.tight_layout()
plt.show()


## 5. Boosting-Specific Attention Analysis

These plots answer the central intervention questions:

1. Does increasing alpha increase attention mass on the intended trace positions?
2. Does trace_all increase THINK attention more strongly than trace_visual?
3. Does higher trace attention coincide with higher accuracy?


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, metric in zip(axes, MASS_METRICS):
    for (target, layer_mode), group in selected_summary.groupby(["boost_target", "layer_mode"]):
        group = group.sort_values("alpha")
        ax.plot(
            group["alpha"], group[metric], marker="o", linewidth=2,
            label=f"{target} / {layer_mode}",
        )
    ax.set_xlabel("Alpha")
    ax.set_ylabel(metric)
    ax.set_xticks(EXPECTED_ALPHAS)
    ax.set_title(metric.replace("_", " "))
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.suptitle("Post-softmax attention mass response to boosting")
plt.tight_layout()
plt.show()


In [ ]:
response_rows = []
for group_key, group in run_summary.groupby(BASE_SETTING_KEYS + ["boost_target", "layer_mode"], dropna=False):
    record = dict(zip(BASE_SETTING_KEYS + ["boost_target", "layer_mode"], group_key))
    record["runs"] = len(group)
    for metric in [*MASS_METRICS, "accuracy_pct"]:
        usable = group[["alpha", metric]].dropna()
        record[f"{metric}_spearman_alpha"] = (
            usable["alpha"].corr(usable[metric], method="spearman")
            if len(usable) >= 2 and usable["alpha"].nunique() >= 2
            else np.nan
        )
        record[f"{metric}_linear_slope"] = (
            float(np.polyfit(usable["alpha"], usable[metric], 1)[0])
            if len(usable) >= 2 and usable["alpha"].nunique() >= 2
            else np.nan
        )
    response_rows.append(record)

alpha_response = pd.DataFrame(response_rows)
display(
    alpha_response.style.format({
        column: "{:.5f}"
        for column in alpha_response.columns
        if column.endswith("_slope") or column.endswith("_alpha")
    }).set_caption("Within-setting response of attention and accuracy to alpha")
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric in zip(axes, MASS_METRICS):
    usable = run_summary.dropna(subset=[metric, "accuracy_pct"])
    for target, group in usable.groupby("boost_target"):
        ax.scatter(group[metric], group["accuracy_pct"], alpha=0.75, label=target)
    correlation = usable[[metric, "accuracy_pct"]].corr(method="spearman").iloc[0, 1] if len(usable) >= 2 else np.nan
    ax.set_xlabel(metric)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"Spearman ρ={correlation:.3f}")
axes[0].legend()
plt.suptitle("Run-level accuracy versus tracked attention mass")
plt.tight_layout()
plt.show()


In [ ]:
target_comparison = (
    run_summary
    .pivot_table(
        index=BASE_SETTING_KEYS + ["layer_mode", "alpha"],
        columns="boost_target",
        values=["accuracy_pct", *MASS_METRICS],
        aggfunc="first",
    )
    .sort_index()
)
display(target_comparison.style.format("{:.5f}").set_caption("Paired trace_all versus trace_visual comparison"))


## 6. Decoding Entropy Analysis

Entropy sidecars retain the original notebook's answer-option and decoded-token uncertainty analyses. The attention-mass fields are also loaded so uncertainty can be compared directly with grounding behavior.


In [ ]:
entropy_rows = []
missing_entropy_files = []

for summary in run_summary.itertuples(index=False):
    entropy_path = resolve_artifact(summary.entropy_tracking_path)
    if entropy_path is None and summary.output_path:
        prediction_path = resolve_artifact(summary.output_path)
        entropy_path = prediction_path.with_name(f"{prediction_path.stem}_entropy_tracking.json")
    if entropy_path is None or not entropy_path.exists():
        missing_entropy_files.append(str(entropy_path))
        continue

    rows = load_json(entropy_path)
    metadata = {key: getattr(summary, key) for key in RUN_KEYS}
    metadata["run_id"] = summary.run_id
    for row in rows:
        entropy_rows.append({
            **metadata,
            "example_id": row.get("example_id"),
            "correct": bool(row.get("correct")),
            "gold_answer": row.get("gold_answer"),
            "decoded_answer": row.get("decoded_answer"),
            "num_output_tokens": row.get("num_output_tokens"),
            "answer_option_entropy": row.get("answer_option_entropy"),
            "decoded_token_entropy_mean": row.get("decoded_token_entropy_mean"),
            "decoded_token_entropy_median": row.get("decoded_token_entropy_median"),
            "decoded_token_entropy_max": row.get("decoded_token_entropy_max"),
            "answer_option_selected_option": row.get("answer_option_selected_option"),
            "answer_option_decoded_token_index": row.get("answer_option_decoded_token_index"),
            "answer_option_probabilities": row.get("answer_option_probabilities"),
            "decoded_token_entropies": row.get("decoded_token_entropies"),
            "trace_attention_mass": row.get("trace_attention_mass"),
            "visual_trace_attention_mass": row.get("visual_trace_attention_mass"),
            "think_attention_mass": row.get("think_attention_mass"),
            "entropy_path": str(entropy_path.relative_to(ROOT)) if entropy_path.is_relative_to(ROOT) else str(entropy_path),
        })

entropy_df = pd.DataFrame(entropy_rows)
for column in [*ENTROPY_METRICS, *MASS_METRICS, "alpha", "num_output_tokens"]:
    entropy_df[column] = pd.to_numeric(entropy_df[column], errors="coerce")

print(f"Entropy rows: {len(entropy_df):,}")
print(f"Runs with entropy: {entropy_df['run_id'].nunique():,}")
print(f"Missing entropy sidecars: {len(missing_entropy_files):,}")
coverage = entropy_df[[*ENTROPY_METRICS, *MASS_METRICS]].notna().mean().rename("fraction_present").to_frame()
display(coverage.style.format("{:.2%}"))


In [ ]:
entropy_summary = (
    entropy_df
    .groupby(RUN_KEYS + ["correct"], dropna=False)
    .agg(
        samples=("example_id", "size"),
        option_entropy_available=("answer_option_entropy", "count"),
        mean_option_entropy=("answer_option_entropy", "mean"),
        median_option_entropy=("answer_option_entropy", "median"),
        mean_answer_entropy=("decoded_token_entropy_mean", "mean"),
        median_answer_entropy=("decoded_token_entropy_mean", "median"),
        mean_peak_token_entropy=("decoded_token_entropy_max", "mean"),
        mean_trace_attention_mass=("trace_attention_mass", "mean"),
        mean_visual_trace_attention_mass=("visual_trace_attention_mass", "mean"),
        mean_think_attention_mass=("think_attention_mass", "mean"),
        mean_output_tokens=("num_output_tokens", "mean"),
    )
    .reset_index()
)
entropy_summary["option_entropy_coverage"] = entropy_summary["option_entropy_available"] / entropy_summary["samples"]
display(
    entropy_summary.style.format({
        "alpha": "{:.1f}",
        "option_entropy_coverage": "{:.2%}",
        **{column: "{:.5f}" for column in entropy_summary.columns if column.startswith("mean_") or column.startswith("median_")},
    }).set_caption("Entropy and attention metrics split by correctness")
)


In [ ]:
selected_entropy_run = entropy_df[entropy_df["run_id"] == SELECTED_RUN_ID].copy()
print(f"Selected entropy rows: {len(selected_entropy_run):,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (metric, title) in zip(axes, [
    ("answer_option_entropy", "Answer-option entropy"),
    ("decoded_token_entropy_mean", "Mean decoded-token entropy"),
]):
    usable = selected_entropy_run.dropna(subset=[metric])
    if HAS_SEABORN:
        sns.histplot(
            data=usable, x=metric, hue="correct", bins=40, kde=True,
            stat="density", common_norm=False, element="step", ax=ax,
        )
    else:
        for correct, group in usable.groupby("correct"):
            ax.hist(group[metric], bins=40, alpha=0.45, density=True, label=f"correct={correct}")
        ax.legend()
    ax.set_title(title)
plt.suptitle("Correct versus incorrect decoding uncertainty")
plt.tight_layout()
plt.show()


### Does Entropy or Attention Predict Errors?

AUC is the probability that a randomly selected incorrect example has a higher metric value than a randomly selected correct example. For attention mass, direction is empirical rather than assumed.


In [ ]:
def error_detection_auc(frame, metric):
    usable = frame[[metric, "correct"]].dropna()
    if usable["correct"].nunique() < 2:
        return np.nan
    error = (~usable["correct"]).astype(int)
    n_error = int(error.sum())
    n_correct = int((1 - error).sum())
    ranks = usable[metric].rank(method="average")
    return float((ranks[error == 1].sum() - n_error * (n_error + 1) / 2) / (n_error * n_correct))


discrimination_rows = []
for run_id, group in entropy_df.groupby("run_id"):
    metadata = group.iloc[0]
    row = {
        **{key: metadata[key] for key in RUN_KEYS},
        "run_id": run_id,
        "samples": len(group),
        "accuracy": group["correct"].mean(),
    }
    for metric in [*ENTROPY_METRICS, *MASS_METRICS]:
        means = group.groupby("correct")[metric].mean()
        row[f"{metric}_error_gap"] = means.get(False, np.nan) - means.get(True, np.nan)
        row[f"{metric}_error_auc"] = error_detection_auc(group, metric)
    discrimination_rows.append(row)

error_discrimination = pd.DataFrame(discrimination_rows).sort_values(RUN_KEYS)
display(
    error_discrimination.style.format({
        "accuracy": "{:.2%}",
        **{
            column: "{:.5f}"
            for column in error_discrimination.columns
            if column.endswith("_gap") or column.endswith("_auc")
        },
    }).set_caption("Error separation from entropy and attention metrics")
)


### Accuracy Across Entropy Quantiles


In [ ]:
quantile_rows = []
for metric in ["answer_option_entropy", "decoded_token_entropy_mean", "decoded_token_entropy_max"]:
    usable = selected_entropy_run.dropna(subset=[metric]).copy()
    if usable[metric].nunique() < 2:
        continue
    usable["metric_quantile"] = pd.qcut(
        usable[metric], q=min(5, usable[metric].nunique()), duplicates="drop"
    )
    grouped = usable.groupby("metric_quantile", observed=True).agg(
        samples=("example_id", "size"),
        accuracy=("correct", "mean"),
        mean_metric=(metric, "mean"),
    ).reset_index()
    grouped["metric"] = metric
    grouped["quantile_rank"] = range(1, len(grouped) + 1)
    grouped["metric_quantile"] = grouped["metric_quantile"].astype(str)
    quantile_rows.append(grouped)

entropy_quantiles = pd.concat(quantile_rows, ignore_index=True) if quantile_rows else pd.DataFrame()
display(entropy_quantiles.style.format({"accuracy": "{:.2%}", "mean_metric": "{:.5f}"}))

if not entropy_quantiles.empty:
    for metric, group in entropy_quantiles.groupby("metric"):
        plt.plot(group["quantile_rank"], group["accuracy"], marker="o", label=metric)
    plt.gca().yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
    plt.xlabel("Entropy quantile (low to high)")
    plt.ylabel("Accuracy")
    plt.title("Accuracy as decoding uncertainty increases")
    plt.legend()
    plt.show()


### Entropy, Attention, and Confidence Relationships


In [ ]:
scatter_metrics = [
    ("answer_option_entropy", "decoded_token_entropy_mean"),
    ("trace_attention_mass", "answer_option_entropy"),
    ("visual_trace_attention_mass", "decoded_token_entropy_mean"),
]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (x_metric, y_metric) in zip(axes, scatter_metrics):
    usable = selected_entropy_run.dropna(subset=[x_metric, y_metric])
    correlation = usable[[x_metric, y_metric]].corr(method="spearman").iloc[0, 1] if len(usable) >= 2 else np.nan
    for correct, group in usable.groupby("correct"):
        ax.scatter(group[x_metric], group[y_metric], alpha=0.6, label=f"correct={correct}")
    ax.set_xlabel(x_metric)
    ax.set_ylabel(y_metric)
    ax.set_title(f"Spearman ρ={correlation:.3f}")
axes[0].legend()
plt.tight_layout()
plt.show()


## 7. Error and Success Examples

These tables surface inference examples with unusually high or low trace attention, confident errors, and uncertain successes. They intentionally avoid mining-time trace features.


In [ ]:
def compact_question(text):
    if not isinstance(text, str):
        return text
    text = re.sub(r"\s+", " ", text).strip()
    return text[:260] + ("..." if len(text) > 260 else "")


inspection = selected_predictions.copy()
inspection["question"] = inspection["question"].map(compact_question)
inspection_cols = [
    "example_id", "domain", "topic", "gold_answer", "decoded_answer",
    "generated_text", "trace_attention_mass", "visual_trace_attention_mass",
    "think_attention_mass", "num_inference_trace_actions", "num_output_tokens", "question",
]

print("Incorrect examples with highest trace attention")
display(inspection[~inspection["correct"]].sort_values("trace_attention_mass", ascending=False)[inspection_cols].head(15))

print("Correct examples with lowest trace attention")
display(inspection[inspection["correct"]].sort_values("trace_attention_mass", ascending=True)[inspection_cols].head(15))

if not selected_entropy_run.empty:
    entropy_inspection = selected_entropy_run.merge(
        selected_predictions[["example_id", "question", "domain", "topic", "generated_text"]],
        on="example_id", how="left",
    )
    print("Most confident errors by answer-option entropy")
    display(
        entropy_inspection[~entropy_inspection["correct"]]
        .sort_values("answer_option_entropy", ascending=True)
        [[
            "example_id", "domain", "topic", "gold_answer", "decoded_answer",
            "answer_option_entropy", "decoded_token_entropy_mean",
            "trace_attention_mass", "generated_text", "question",
        ]]
        .head(15)
    )


## 8. Domain and Topic Breakdown

Accuracy and attention mass are summarized across M3CoT domains and topics for every boost configuration.


In [ ]:
domain_summary = (
    predictions.dropna(subset=["domain"])
    .groupby(RUN_KEYS + ["domain"], dropna=False)
    .agg(
        examples=("example_id", "nunique"),
        accuracy=("correct", "mean"),
        mean_trace_attention_mass=("trace_attention_mass", "mean"),
        mean_visual_trace_attention_mass=("visual_trace_attention_mass", "mean"),
        mean_think_attention_mass=("think_attention_mass", "mean"),
    )
    .reset_index()
)
domain_summary["accuracy_pct"] = 100 * domain_summary["accuracy"]
display(
    domain_summary.sort_values(["accuracy", "examples"], ascending=[False, False]).head(60)
    .style.format({
        "alpha": "{:.1f}", "accuracy": "{:.4f}", "accuracy_pct": "{:.2f}",
        "mean_trace_attention_mass": "{:.5f}",
        "mean_visual_trace_attention_mass": "{:.5f}",
        "mean_think_attention_mass": "{:.5f}",
    })
)


In [ ]:
topic_summary = (
    predictions.dropna(subset=["topic"])
    .groupby(RUN_KEYS + ["topic"], dropna=False)
    .agg(
        examples=("example_id", "nunique"),
        accuracy=("correct", "mean"),
        mean_trace_attention_mass=("trace_attention_mass", "mean"),
        mean_visual_trace_attention_mass=("visual_trace_attention_mass", "mean"),
        mean_think_attention_mass=("think_attention_mass", "mean"),
    )
    .reset_index()
)
topic_summary["accuracy_pct"] = 100 * topic_summary["accuracy"]

selected_domain = domain_summary[
    (domain_summary["run_type"] == selected_summary.iloc[0]["run_type"])
    & (domain_summary["mined_by"] == selected_summary.iloc[0]["mined_by"])
    & (domain_summary["context"] == selected_summary.iloc[0]["context"])
    & (domain_summary["evaluated_by"] == selected_summary.iloc[0]["evaluated_by"])
    & (domain_summary["trace_variant"] == selected_summary.iloc[0]["trace_variant"])
]
if not selected_domain.empty:
    chart = (
        selected_domain.groupby(["domain", "boost_target", "layer_mode"], dropna=False)
        .agg(accuracy=("accuracy", "mean"))
        .reset_index()
    )
    chart["setting"] = chart["boost_target"] + " / " + chart["layer_mode"]
    pivot = chart.pivot(index="domain", columns="setting", values="accuracy")
    pivot.plot(kind="bar", figsize=(13, 5))
    plt.gca().yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
    plt.ylabel("Mean accuracy over alpha sweep")
    plt.title("Selected base setting: domain accuracy by boost configuration")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

display(topic_summary.sort_values(["accuracy", "examples"], ascending=[False, False]).head(60))


## 9. Export Analysis Tables

Set `EXPORT_TABLES = True` to save inference-only tables for reporting or further analysis.


In [ ]:
EXPORT_TABLES = False
EXPORT_DIR = ROOT / "analysis/test_trace_answer_boosted_analysis_tables"

if EXPORT_TABLES:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    run_summary.to_csv(EXPORT_DIR / "boosted_run_summary.csv", index=False)
    sweep_coverage.to_csv(EXPORT_DIR / "sweep_coverage.csv", index=False)
    ranked_runs.to_csv(EXPORT_DIR / "ranked_runs.csv", index=False)
    best_per_base.to_csv(EXPORT_DIR / "best_setting_per_base.csv", index=False)
    predictions.to_csv(EXPORT_DIR / "predictions_by_sample.csv", index=False)
    correctness_summary.to_csv(EXPORT_DIR / "correctness_summary.csv", index=False)
    alpha_response.to_csv(EXPORT_DIR / "alpha_response.csv", index=False)
    entropy_df.drop(
        columns=["answer_option_probabilities", "decoded_token_entropies"], errors="ignore"
    ).to_csv(EXPORT_DIR / "entropy_by_sample.csv", index=False)
    entropy_summary.to_csv(EXPORT_DIR / "entropy_summary.csv", index=False)
    error_discrimination.to_csv(EXPORT_DIR / "error_discrimination.csv", index=False)
    domain_summary.to_csv(EXPORT_DIR / "domain_summary.csv", index=False)
    topic_summary.to_csv(EXPORT_DIR / "topic_summary.csv", index=False)
    print(f"Wrote analysis tables to {EXPORT_DIR}")
else:
    print("Set EXPORT_TABLES = True to write CSV outputs.")
